# Reviewer notebook: cluster-product dynamics

The historical CAQD title is retained. This notebook explores sampled residual estimates and dense numerical reference errors. The analytic actual-defect bound is conditional; these runs are **HEURISTIC_UNCERTIFIED** and do not bound numerical integration or quadrature error.

The product retraction differential is marginal preserving on the stated domain. Its norm witness does not establish best tangent approximation. See manuscript sections 2-4 and the claim ledger.


In [ ]:
import sys
from pathlib import Path

# Add production code directory to path
paper_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code/caqd/__init__.py").is_file()), None)
if paper_root is None:
    raise RuntimeError("Run from the package root or its notebook directory")
code_dir = str(paper_root / "code")
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

import numpy as np
import matplotlib.pyplot as plt

# Production CAQD imports
from caqd.spin_chains import (
    TFIMChain,
    XXZChain,
    neel_state,
    polarized_x_state,
    staggered_magnetization,
    center_correlation,
    transverse_magnetization,
    operator_norm,
)
from caqd.dynamics import evolve_exact, evolve_mean_field
from caqd.adaptive import evolve_adaptive
from caqd.bounds import verify_bound_honesty, compute_cumulative_bounds

print("CAQD package loaded successfully from:", code_dir)


In [ ]:
# Algebraic differential identity at unit-trace full-rank factors.
from caqd.residuals import retraction_tangent_projector
a, b = np.diag([0.8, 0.2]), np.diag([0.7, 0.3])
z = np.diag([1, 0, 0, -1]).astype(complex)
pz = retraction_tangent_projector(a, b, z)
assert np.allclose(retraction_tangent_projector(a, b, pz), pz)
assert np.allclose(pz.reshape(2,2,2,2).trace(axis1=1, axis2=3), z.reshape(2,2,2,2).trace(axis1=1, axis2=3))


### Editable parameters
Change the Python arguments below to select TFIM/XXZ, chain size, observable, and heuristic target. There are no slider widgets. The baseline uses a nonzero center-correlation error; max_bond is a cluster-qubit ceiling.


In [ ]:
def run_simulation(model_name="TFIM", N=4, J=1.0, field=1.0, obs_name="C_mid", tol=0.1, t_max=0.4):
    """Execute exact, mean-field, and adaptive CAQD simulations."""
    times = np.linspace(0.0, t_max, 25)
    
    # Instantiate Model
    if model_name == "TFIM":
        model = TFIMChain(num_sites=N, J=J, h=field)
        rho0 = polarized_x_state(N)
    else:
        model = XXZChain(num_sites=N, J=J, delta=1.0, h=field)
        rho0 = neel_state(N)
        
    # Select Observable
    if obs_name == "M_stag":
        obs = staggered_magnetization(N)
    elif obs_name == "C_mid":
        obs = center_correlation(N)
    else:
        obs = transverse_magnetization(N)
        
    obs_dict = {obs_name: obs}
    
    # Run exact unitary benchmark
    exact_res = evolve_exact(model, rho0, times, obs_dict)
    
    # Run fixed product mean-field (chi=1)
    mf_res = evolve_mean_field(model, rho0, times, obs_dict)
    
    # Run heuristic adaptive cluster dynamics (CAQD)
    adapt_res = evolve_adaptive(model, rho0, obs_dict, times, tol=tol, max_bond=4, refine_threshold=0.03)
    
    # Compute bounds and verification report
    mf_report = verify_bound_honesty(exact_res, mf_res, obs_dict)
    adapt_report = verify_bound_honesty(exact_res, adapt_res, obs_dict)
    
    return {
        "times": times,
        "model": model,
        "obs_name": obs_name,
        "obs": obs,
        "exact_res": exact_res,
        "mf_res": mf_res,
        "adapt_res": adapt_res,
        "mf_report": mf_report[obs_name],
        "adapt_report": adapt_report[obs_name],
    }

# Execute baseline run
sim_data = run_simulation(model_name="TFIM", N=4, J=1.0, field=1.0, obs_name="C_mid", tol=0.1, t_max=0.4)
print("Simulation complete. Exact, Mean-Field, and Adaptive trajectories ready for inspection.")

# Numerical regression observations, not certification.
assert sim_data["adapt_res"].metadata["certificate_status"] == "HEURISTIC_UNCERTIFIED"
assert np.all(np.diff(sim_data["adapt_report"].certified_bounds) >= -1e-12)
assert sim_data["mf_report"].true_errors.max() > 0.1
for result in [sim_data["exact_res"], sim_data["mf_res"], sim_data["adapt_res"]]:
    for rho in result.states:
        assert np.allclose(rho, rho.conj().T, atol=1e-10)
        assert np.isclose(np.trace(rho), 1, atol=1e-10)
        assert np.linalg.eigvalsh(rho).min() >= -1e-10
print("Adaptive estimated target met:", sim_data["adapt_res"].metadata["estimated_target_met"])


In [ ]:
# Plot 1: Observable Trajectories
times = sim_data["times"]
obs_name = sim_data["obs_name"]
exact_exp = sim_data["exact_res"].expectation(obs_name)
mf_exp = sim_data["mf_res"].expectation(obs_name)
adapt_exp = sim_data["adapt_res"].expectation(obs_name)

plt.figure(figsize=(8, 5))
plt.plot(times, exact_exp, "k-", linewidth=2.5, label=r"Dense unitary reference")
plt.plot(times, mf_exp, "r--", linewidth=2, label=r"Product Mean-Field ($\chi=1$)")
plt.plot(times, adapt_exp, "b-.", linewidth=2, label=r"Adaptive prototype")
plt.xlabel("Time $t$", fontsize=12)
plt.ylabel(f"Expectation $\\langle {obs_name} \\rangle(t)$", fontsize=12)
plt.title(f"Observable Dynamics: Exact vs Mean-Field vs Adaptive CAQD ({obs_name})", fontsize=13)
plt.legend(frameon=True, fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Sampled estimates and reference comparison
mf_v = sim_data["mf_report"]
adapt_v = sim_data["adapt_report"]

plt.figure(figsize=(8, 5))
plt.plot(times, mf_v.true_errors, "r-", label=r"Mean-field reference error $|\Delta O(t)|$")
plt.plot(times, mf_v.certified_bounds, "r--", alpha=0.7, label=r"Mean-field sampled estimate $B_O^{MF}(t)$")
plt.plot(times, adapt_v.true_errors, "b-", label=r"Adaptive reference error $|\Delta O_{adapt}(t)|$")
plt.plot(times, adapt_v.certified_bounds, "b--", alpha=0.7, label=r"Adaptive sampled estimate $B_O^{adapt}(t)$")

plt.xlabel("Time $t$", fontsize=12)
plt.ylabel("Reference error and sampled estimate", fontsize=12)
plt.title("Sampled estimates versus numerical reference errors", fontsize=13)
plt.legend(frameon=True, fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Mean-Field Minimum Margin: {mf_v.min_margin:.6e} (Sampled margin nonnegative: {mf_v.is_honest})")
print(f"Adaptive Minimum Margin:   {adapt_v.min_margin:.6e} (Sampled margin nonnegative: {adapt_v.is_honest})")


In [ ]:
# Plot 3: Spacetime Heatmap of Local Residual Rates nu_e(t)
model = sim_data["model"]
mf_res = sim_data["mf_res"]

edges = model.edges
rate_matrix = np.array([mf_res.residuals[e] for e in edges]) # shape: (num_edges, num_times)

plt.figure(figsize=(9, 4))
im = plt.imshow(
    rate_matrix,
    aspect="auto",
    origin="lower",
    extent=[times[0], times[-1], -0.5, len(edges) - 0.5],
    cmap="inferno",
)
plt.colorbar(im, label=r"Defect Rate $\nu_e(t) = \|n_e(t)\|_1$")
plt.yticks(range(len(edges)), [f"Edge {e}" for e in edges])
plt.xlabel("Time $t$", fontsize=12)
plt.ylabel("Lattice Bond", fontsize=12)
plt.title("Spacetime Heatmap of Instantaneous Projection Defect Rates", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 4: Cluster Size / Cluster-qubit ceiling in Adaptive CAQD
adapt_res = sim_data["adapt_res"]
cluster_hist = adapt_res.metadata.get("cluster_history", [])

# Compute max cluster size over time
max_cluster_sizes = [max(len(c) for c in snapshot) for snapshot in cluster_hist]

plt.figure(figsize=(8, 4))
plt.step(times, max_cluster_sizes, "g-", where="post", linewidth=2.5, label=r"Max Cluster Size $K(t)$")
plt.xlabel("Time $t$", fontsize=12)
plt.ylabel("Maximum Cluster Size (qubits)", fontsize=12)
plt.title("Dynamic Variational Subspace Allocation Driven by Residuals", fontsize=13)
plt.ylim(0.5, model.num_sites + 0.5)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print("Final cluster configuration:", adapt_res.metadata.get("final_clusters"))


### Reproduction and formal scope
- Replay: `python scripts/replay_notebook.py` and `python scripts/replay_notebook.py --from-notebook-dir` from the package root. These execute code cells headlessly, not a live notebook UI.
- Suite and archive: `python verify_deposit.py --skip-lean` checks pytest with at least 95% coverage, metadata, hashes, archive inventory, observations, and notebook replay. It explicitly reports Lean skipped.
- Lean sources establish conditional Nat/Int finite-sum inequalities using supplied mathematical hypotheses. They do not model density matrices, continuous Duhamel integrals, or the solver. The stored axiom report is historical; a fresh audit requires the pinned toolchain. See `lean4/README.md`.
